# Ch3 심화 — 하네스를 직접 열어 조립해 본다 (`create_deep_agent`)

본문 `--trace`가 하던 introspection을 셀 단위로 직접 해봅니다. **키 불필요** — 하네스를 *빌드*하는 것만으로 모델을 호출하지 않습니다.

두 가지를 봅니다. ① 우리가 도구 하나만 넘겨도 하네스가 나머지를 자동 배선한다는 것, ② `agent.invoke(...)`가 돌려주는 `out`이 실은 **최종 상태(state)** 라는 것.

## 실험 1 — 우리가 넘긴 것 vs 하네스가 채운 것

`list_records` 도구 하나 + 서브에이전트 명세만 넘기고, **빌드된 그래프에서 실제 배선된 도구·미들웨어를 꺼냅니다.**

In [1]:
import sys, pathlib
here = pathlib.Path.cwd()
for p in [here, here / "ch3-deepagents", here.parent]:      # 노트북 위치/실행 위치 어디서든 소스를 찾는다
    if (p / "research_orchestrator.py").exists():
        sys.path.insert(0, str(p)); break

from research_orchestrator import SUBAGENT_SPECS, ORCHESTRATOR_PROMPT, LIVE_MODEL, OPENROUTER_BASE_URL
from deepagents import create_deep_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI


@tool
def list_records() -> str:
    """분류 레코드를 JSON으로(스텁 — 여기선 호출되지 않음)."""
    return "[]"


# 키 없이 '구성만': ChatOpenAI 생성도 create_deep_agent 빌드도 모델을 호출하지 않는다.
llm = ChatOpenAI(model=LIVE_MODEL.removeprefix("openai:"), base_url=OPENROUTER_BASE_URL, api_key="no-call")
agent = create_deep_agent(model=llm, tools=[list_records],
                          system_prompt=ORCHESTRATOR_PROMPT, subagents=SUBAGENT_SPECS)

wired = list(agent.nodes["tools"].bound.tools_by_name)     # 빌드된 그래프에서 실제 도구를 꺼낸다
added = [t for t in wired if t != "list_records"]
print("우리가 넘긴 도구      :", ["list_records"])
print("하네스가 자동 배선한 도구:", added, f"({len(added)}개)")
print("그래프 노드          :", list(agent.get_graph().nodes))
print("미들웨어             :", [n for n in agent.get_graph().nodes if "Middleware" in n])

우리가 넘긴 도구      : ['list_records']
하네스가 자동 배선한 도구: ['write_todos', 'ls', 'read_file', 'write_file', 'edit_file', 'glob', 'grep', 'execute', 'task'] (9개)
그래프 노드          : ['__start__', 'model', 'tools', 'TodoListMiddleware.after_model', 'PatchToolCallsMiddleware.before_agent', '__end__']
미들웨어             : ['TodoListMiddleware.after_model', 'PatchToolCallsMiddleware.before_agent']


**관찰.** 도구 1개를 넘겼는데 `write_todos`(계획)·`task`(위임)·파일시스템·`execute`까지 배선됩니다. 이게 '배터리 포함 하네스'입니다.

### ✏️ 직접 해보기

`SUBAGENT_SPECS`에 네 번째 워커를 하나 더해 다시 빌드해 보세요. 아래 빈칸(`___`)을 채우고 실행하면, `task`가 위임할 대상이 늘어난 걸 확인합니다.

In [2]:
extra = {
    "name": "___",                      # 예: "tax_check"
    "description": "___",               # 이 워커가 무슨 대사를 하나 한 줄
    "system_prompt": "___",             # 워커에게 줄 지시(짧게)
}
agent2 = create_deep_agent(model=llm, tools=[list_records],
                           system_prompt=ORCHESTRATOR_PROMPT,
                           subagents=SUBAGENT_SPECS + [extra])
# 서브에이전트는 task 도구 하나로 위임된다 — 목록이 늘어도 상위 도구 수는 그대로다. 무엇이 그대로이고
# 무엇이 바뀌는지 아래로 확인하자.
print("서브에이전트 수:", len(SUBAGENT_SPECS) + 1)
print("상위 도구       :", [t for t in agent2.nodes["tools"].bound.tools_by_name])

서브에이전트 수: 4
상위 도구       : ['write_todos', 'ls', 'read_file', 'write_file', 'edit_file', 'glob', 'grep', 'execute', 'task', 'list_records']


## 실험 2 — `agent.invoke(...)`의 `out`은 무엇인가

`out = agent.invoke({"messages": [...]})`. 흔히 착각하는데 `out`은 *마지막 답 하나*가 아니라 그래프가 끝난 뒤의 **최종 상태(state) dict**입니다. 어떤 키가 있는지 키 없이 스키마로 봅니다.

In [3]:
schema = agent.get_output_jsonschema()          # 호출 없이 '출력 상태'의 스키마
print("out(=최종 상태)의 키:", sorted(schema["properties"]))

print("""
  messages            : 전체 대화 기록(내 프롬프트 · AI의 tool_calls · ToolMessage · 최종 답)
                        → fan_out_live가 바로 이걸 훑어 'write_todos 했나 · task 3개 함께 던졌나'를 검증한다
  todos               : write_todos가 남긴 계획 상태 (TodoListMiddleware)
  files               : write_note/write_file가 쓴 가상 파일시스템 상태
  structured_response : response_format를 줬을 때만 채워진다(이 실습에선 안 씀)
""")
print("그래서 코드의 out[\"messages\"][-1].content = 마지막 메시지 = 모델의 최종 요약,")
print("       tool_call_names(out[\"messages\"]) = 그 기록을 훑어 어떤 도구가 불렸는지 감사.")

out(=최종 상태)의 키: ['files', 'messages', 'structured_response', 'todos']

  messages            : 전체 대화 기록(내 프롬프트 · AI의 tool_calls · ToolMessage · 최종 답)
                        → fan_out_live가 바로 이걸 훑어 'write_todos 했나 · task 3개 함께 던졌나'를 검증한다
  todos               : write_todos가 남긴 계획 상태 (TodoListMiddleware)
  files               : write_note/write_file가 쓴 가상 파일시스템 상태
  structured_response : response_format를 줬을 때만 채워진다(이 실습에선 안 씀)

그래서 코드의 out["messages"][-1].content = 마지막 메시지 = 모델의 최종 요약,
       tool_call_names(out["messages"]) = 그 기록을 훑어 어떤 도구가 불렸는지 감사.


**정리.** `out`은 출력 하나가 아니라 **상태**입니다. `fan_out_live`가 어려워 보이는 이유도 여기 있습니다 — 그 함수는 `out["messages"]`(대화 기록)를 훑어 **하네스가 설계대로 동작했는지 네 가지를 검증**합니다: ① `write_todos`로 계획했나 ② 기대한 세 서브에이전트를 `task`했나 ③ 셋을 **같은 턴**에 던졌나(진짜 fan-out) ④ 세 노트가 실제로 저장됐나. 하나라도 어긋나면 통과시키지 않고 멈춥니다(fail-closed). '검증자가 병목'이라는 이 챕터 명제가 코드로 박힌 자리입니다.